*Note: Code has been heavily borrowed from Qiao Jin's GitHub, creator of MedCPT. All relevant credit goes to Qiao Jin.* https://github.com/ncbi/MedCPT

---

# MedCPT

---

In [42]:
# Load in libraries
from pathlib import Path
import numpy as np
import pandas as pd 
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import faiss

In [2]:
CONDITION = "aig"
DATA_DIR = Path(f"../pmc/{CONDITION}/processed")
BATCH_SIZE = 8
QUERY = "diagnostic criteria for autoimmune gastritis"


MedCPT expects a title and absract, so we need to merge titles and with the text 

In [3]:
# read in the articles
articles = pd.read_parquet(DATA_DIR/"articles.parquet")

# read in the passages
passages = pd.read_parquet(DATA_DIR/"passages.parquet")

In [4]:
print("Articles:", articles.columns.tolist())
print("Passages:", passages.columns.tolist())

Articles: ['pmcid', 'title', 'year', 'doi', 'pmid', 'license']
Passages: ['pmcid', 'passage_id', 'year', 'section', 'type', 'text', 'word_count']


In [5]:
# add each paper's title to its passages
passages = passages.merge(articles[["pmcid", "title"]],
                        on="pmcid",
                        how = "left")

In [6]:
# Let's clean our data

# Keep passages containing at least one non-whitespace character
passages = passages[
    passages["text"].str.contains(r"\S", na=False)].copy()

# Replace missing article titles with empty strings
passages["title"] =  passages["title"].fillna("")

In [7]:
# each document contains the title and a passage
# this follows the title-abstract format expected by MedCPT but
# with passages
docs = passages[["title", "text"]].values.tolist()

# Print the first five examples
for doc in docs[:5]:
    print(doc)
    print()

['Autoimmune gastritis with a gastric hamartomatous inverted polyp and two hyperplastic polyps: a case report', 'We report an unusual case of autoimmune gastritis (AIG) complicated with a submucosal tumor (SMT) and two pedunculated polyps in a 60-year-old man. The patient was admitted for epigastric distention, heartburn, and anorexia. Endoscopy showed an SMT in the fundus, two pedunculated polyps in the body, and markedly atrophic mucosa of the body and fundus. The SMT, measuring 20 mm in diameter, was resected by endoscopic submucosal dissection and histologically diagnosed as a gastric hamartomatous inverted polyp (GHIP), which is characterized by submucosal glandular proliferation, cystic dilatation, and calcification. The gland structures consisted of foveolar cells and pseudopyloric or mucous-neck cell types. The two pedunculated polyps that were resected by endoscopic mucosal resection were histologically diagnosed as hyperplastic polyps, which are characterized by hyperplastic 

## MedCPT Cross Encoder

In [8]:
# Load the MedCPT Cross Encoder
model_name = "ncbi/MedCPT-Cross-Encoder"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Use a GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

In [9]:
print(model.config.max_position_embeddings)

512


In [10]:
# Use the passage text as the document
passages = passages["text"].tolist()
print(passages[:5])

# Combine the query and each passage into pairs
pairs = [[QUERY, passages] for passages in passages]

print(pairs[:5])

['We report an unusual case of autoimmune gastritis (AIG) complicated with a submucosal tumor (SMT) and two pedunculated polyps in a 60-year-old man. The patient was admitted for epigastric distention, heartburn, and anorexia. Endoscopy showed an SMT in the fundus, two pedunculated polyps in the body, and markedly atrophic mucosa of the body and fundus. The SMT, measuring 20 mm in diameter, was resected by endoscopic submucosal dissection and histologically diagnosed as a gastric hamartomatous inverted polyp (GHIP), which is characterized by submucosal glandular proliferation, cystic dilatation, and calcification. The gland structures consisted of foveolar cells and pseudopyloric or mucous-neck cell types. The two pedunculated polyps that were resected by endoscopic mucosal resection were histologically diagnosed as hyperplastic polyps, which are characterized by hyperplastic foveolar glands with pseudopyloric or mucous-neck glands in the inflamed stroma in the mucosa, which consisted 

In [11]:
# Store the relevance scores from each batch
all_scores = []

# Score all passages in batches
with torch.no_grad():

    for start in tqdm(range(0, len(pairs), BATCH_SIZE)):

        # Select the next batch of query-passage pairs
        batch = pairs[start:start + BATCH_SIZE]

        # Convert the pairs into tokens
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        ).to(device)

        # Calculate the relevance scores
        logits = model(**encoded).logits.squeeze(dim=1)

        # Move scores to the CPU and store them
        all_scores.append(logits.cpu().numpy())

# Combine all scores
scores = np.concatenate(all_scores)

  0%|          | 0/517 [00:00<?, ?it/s]

In [15]:
# Check that the number of passages matches the scores
print(len(passages), len(scores))


4131 4131


In [16]:
# Add the relevance scores
results = passages.copy()
results["score"] = scores

# Rank passages from most to least relevant
results = results.sort_values("score", ascending=False)

# Display the 10 highest-ranked passages
display(results[["pmcid", "passage_id", "section", "score", "text"]].head(10))

,pmcid,passage_id,section,score,text
204,PMC10378041,3,1. Introduction,15.983333,There are several challenges in diagnosing aut...
328,PMC10496934,1,Abstract,15.962286,Autoimmune gastritis is characterized by infla...
419,PMC10496934,92,Conclusions,15.958878,Autoimmune gastritis is a complex condition ch...
227,PMC10378041,26,5. Diagnosis of Autoimmune Gastritis > 5.1. Ov...,15.874925,"Currently, internationally standardized diagno..."
978,PMC11863545,25,Discussion,15.826061,"Recently, the “Study Group on the Establishmen..."
353,PMC10496934,26,Review,15.730204,Diagnosing autoimmune gastritis involves a com...
360,PMC10496934,33,Review,15.716096,Subclinical and early-stage disease: Autoimmun...
334,PMC10496934,7,Review,15.667618,A comprehensive literature search was conducte...
364,PMC10496934,37,Review,15.659933,Overlapping comorbidities: Autoimmune gastriti...
782,PMC11354099,2,1. Introduction,15.597147,Chronic atrophic autoimmune gastritis (CAAG) i...


## MedCPT Article Encoder

In [17]:
model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")
tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [21]:
# Use a GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the correct device
model = model.to(device).eval()

all_embeddings_article_encoder = []

In [22]:
# Generate embeddings for all passages
with torch.no_grad():

    for start in tqdm(range(0, len(docs), BATCH_SIZE)):

        # Select the next batch of passages
        batch = docs[start:start + BATCH_SIZE]

        # Convert the title-passage pairs into tokens
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        ).to(device)

        # Use the [CLS] token as the passage representation
        embeds = model(**encoded).last_hidden_state[:, 0, :]

        # Move embeddings back to the CPU and store them
        all_embeddings_article_encoder.append(embeds.cpu().numpy())

# Combine all batches into one array
passage_embeddings = np.vstack(all_embeddings_article_encoder )

print(passage_embeddings.shape)

  0%|          | 0/517 [00:00<?, ?it/s]

(4131, 768)


In [23]:
print(model.config._name_or_path)
print(next(model.parameters()).device)

ncbi/MedCPT-Article-Encoder
cuda:0


In [24]:
import umap
import plotly.express as px

# Reduce the embeddings to two dimensions
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    metric="cosine",
    random_state=42
)

coords = reducer.fit_transform(passage_embeddings)

/home/cfernan7/Documents/SCI501/.venv/lib64/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/home/cfernan7/Documents/SCI501/.venv/lib64/python3.12/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [34]:
# Create a DataFrame for visualisation
plot_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "pmcid": passages["pmcid"].values,
    "section": passages["section"].astype(str).values,
    "text": passages["text"].str[:300].values
})

# Plot the passage embeddings
fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    hover_data=["pmcid", "section", "text"],
    title="MedCPT Article Encoder - Passage Embeddings",
    opacity=0.6
)

# Set the marker size
fig.update_traces(marker=dict(size=5))

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## MedCPT Query Encoder

In [36]:
# Load the MedCPT Query Encoder
query_name = "ncbi/MedCPT-Query-Encoder"

query_tokenizer = AutoTokenizer.from_pretrained(query_name)
query_model = AutoModel.from_pretrained(query_name).to(device).eval()

query = ["What are the diagnostic criteria for autoimmune gastritis?"]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# Convert the query into tokens
inputs = query_tokenizer(
    query,
    truncation = True,
    padding = True,
    max_length=64,
    return_tensors="pt"
)

In [38]:
# Convert the query into tokens
inputs = query_tokenizer(
    query,
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
).to(device)

# Generate the query embedding
with torch.no_grad():
    query_embedding = query_model(**inputs).last_hidden_state[:, 0, :]

print(query_embedding.shape)

torch.Size([1, 768])


In [ ]:
# Convert passage embeddings to float32 for FAISS
article_vectors = np.ascontiguousarray(
    passage_embeddings.astype("float32")
)

# Convert query embedding to float32
query_vectors = np.ascontiguousarray(
    query_embedding.cpu().numpy().astype("float32")
)

# Create an inner-product search index per Jin
index = faiss.IndexFlatIP(article_vectors.shape[1])

# Add the passage embeddings to the index
index.add(article_vectors)

# Find the 10 most relevant passages
scores, indices = index.search(query_vectors, k=10)

# Get the matching passages from the original DataFrame
top_passages = passages.iloc[indices[0]].copy()

# Add their relevance scores
top_passages["score"] = scores[0]

# show the results
display(top_passages[["pmcid", "passage_id", "section", "score", "text"]])

,pmcid,passage_id,section,score,text
3331,PMC8621657,20,2. Diagnostic Approaches,70.686089,There is no gold standard for the diagnosis of...
3337,PMC8621657,26,2. Diagnostic Approaches > 2.2. Endoscopic Exa...,70.447708,The diagnostic value of endoscopic signs of CA...
3339,PMC8621657,28,2. Diagnostic Approaches > 2.3. Morphological ...,69.724121,Histological examination of gastrobiopsy speci...
3338,PMC8621657,27,2. Diagnostic Approaches > 2.2. Endoscopic Exa...,69.608055,According to the Guidelines for the Management...
3320,PMC8621657,9,1. Introduction > 1.2. Clinical Manifestation,69.466377,CAG can be asymptomatic or can be characterize...
1996,PMC3520153,8,2. Patients and Methods,69.347687,The cut-off values adopted were those suggeste...
615,PMC11010983,5,2. Epidemiology,69.326569,The prevalence of AAG in the general populatio...
959,PMC11863545,6,Patients and methods,69.230804,The objective of this retrospective cohort stu...
2656,PMC6502205,16,Conclusion,69.181526,"In practical terms, it is advisable to screen ..."
627,PMC11010983,17,4. Diagnosis,69.167908,An accurate endoscopic examination is mandator...
